# Training a Character-Level Language Model

**Module 5** — Language Models & Sequence Generation

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

Here's the full architecture: each character is converted to a one-hot vector (or looked up in an embedding table), fed through an LSTM, and the LSTM's output is passed through a linear layer + softmax to produce a probability distribution over all possible next characters.

> **Note:** Why start with characters? Because the vocabulary is tiny — just 26 letters, spaces, and punctuation. A word-level model needs 50,000+ vocabulary entries. Characters let us focus on the *mechanism* without drowning in vocabulary management.


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn

class CharLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        # x: (batch, seq_len) of character indices
        emb = self.embedding(x)            # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)  # (batch, seq_len, hidden_dim)
        logits = self.fc(out)              # (batch, seq_len, vocab_size)
        return logits, hidden

### Step 2: Training setup


In [ ]:
vocab = sorted(set('hello world, this is a character language model!'))
char_to_idx = {c: i for i, c in enumerate(vocab)}
vocab_size = len(vocab)

model = CharLM(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### Step 3: The key idea: input is chars[:-1], target is chars[1:]


In [ ]:
text = 'hello world, this is a character language model!'
indices = [char_to_idx[c] for c in text]
input_seq = torch.tensor(indices[:-1]).unsqueeze(0)   # all but last
target_seq = torch.tensor(indices[1:]).unsqueeze(0)   # all but first

for epoch in range(100):
    logits, _ = model(input_seq)
    loss = criterion(logits.squeeze(0), target_seq.squeeze(0))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

## Key Takeaway

The crucial pattern: **input is the sequence shifted right by one, target is the sequence shifted left by one**. At every position t, the model sees characters 1 through t and must predict character t+1. The LSTM's hidden state accumulates all previous context.

*A complete character-level LSTM language model in PyTorch*

---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
